# Supplementary Analyses — Diabetes Risk Screening

Four checks from the course method that the main notebooks did not cover explicitly. They were added while
preparing the extended abstract (decisions D-82 to D-86 in `DECISIONS.md`, results in `RESULTS.md`, step 13).

| Section | Course topic | Question |
|---|---|---|
| 1. Confusion matrix | Evaluation | How many people does the live model catch, miss and flag falsely? |
| 2. Outliers | EDA (Z-score, IQR) | Are there extreme values in the numeric features, and are they errors? |
| 3. Feature selection | ANOVA F-test (`f_classif`) | Which features matter most, and would fewer questions be enough? |
| 4. Hyperparameter tuning | GridSearchCV + cross-validation | Does tuning CatBoost improve on the default settings (D-21)? |

Same data, target and 80/20 split (seed 42) as `Modeling.ipynb` and `ml/train.py`. Everything except section 1's
final numbers uses **only the training rows**; the test set is used once per model, as before (D-24).

> **Runtime:** about 1–1.5 hours, almost all of it in sections 3.1 and 4 (CatBoost cross-validation).

## 0. Setup

In [ ]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from catboost import CatBoostClassifier
from sklearn.feature_selection import f_classif
from sklearn.metrics import (ConfusionMatrixDisplay, confusion_matrix, roc_auc_score, average_precision_score,
                             precision_recall_curve, recall_score, precision_score, f1_score, accuracy_score)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split

SEED = 42
N_FOLDS = 5
TARGET_RECALL = 0.80
BASE_PARAMS = {"iterations": 500, "learning_rate": 0.1, "depth": 6}   # D-21, same as ml/train.py
ROOT = Path("../..")
plt.rcParams["figure.figsize"] = (9, 4)

df = pd.read_csv(ROOT / "data/raw_data/diabetes_012_health_indicators_BRFSS2015.csv").astype(int)
y = (df["Diabetes_012"] > 0).astype(int).rename("diabetes")
X = df.drop(columns="Diabetes_012")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)

# the same rows as the shared split file (D-32)
split = pd.read_csv(ROOT / "data/splits/train_test_split.csv")
assert set(split.loc[split["split"] == "test", "row"]) == set(X_test.index)
print(f"Train: {len(X_train):,} rows ({y_train.mean():.1%} positive), test: {len(X_test):,} rows ({y_test.mean():.1%})")

cv = StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED)

def make_model(params=None, **extra):
    return CatBoostClassifier(**(params or BASE_PARAMS), random_seed=SEED, verbose=0,
                              allow_writing_files=False, **extra)

def threshold_for_recall(y_true, proba, target=TARGET_RECALL):
    precision, recall, thresholds = precision_recall_curve(y_true, proba)
    return float(thresholds[np.where(recall[:-1] >= target)[0].max()])

def oof_proba(params, cols):
    """Out-of-fold predictions on the training rows (5-fold CV), as in ml/train.py."""
    oof = np.zeros(len(y_train))
    for tr, va in cv.split(X_train, y_train):
        m = make_model(params).fit(X_train.iloc[tr][cols], y_train.iloc[tr])
        oof[va] = m.predict_proba(X_train.iloc[va][cols])[:, 1]
    return oof

def summary(y_true, proba, threshold=None):
    precision, recall, _ = precision_recall_curve(y_true, proba)
    out = {"ROC-AUC": roc_auc_score(y_true, proba), "PR-AUC": average_precision_score(y_true, proba),
           "Precision @ 80% recall": float(precision[recall >= TARGET_RECALL].max())}
    if threshold is not None:
        flagged = proba >= threshold
        out.update({"Threshold": threshold, "Recall": recall_score(y_true, flagged),
                    "Precision": precision_score(y_true, flagged), "F1": f1_score(y_true, flagged),
                    "Accuracy": accuracy_score(y_true, flagged), "Flagged": flagged.mean()})
    return out

## 1. Confusion matrix of the live model

The model that is live in the app (`models/latest.txt`), with **its own saved threshold** (80% recall, chosen on
out-of-fold predictions, D-23), on the 50,736 test rows. This gives the exact counts behind the "caught / missed /
false alarms" table in `RESULTS.md` (step 2), which were derived from rounded percentages.

In [ ]:
run_id = (ROOT / "models/latest.txt").read_text().strip()
meta = json.loads((ROOT / "models" / run_id / "metadata.json").read_text())
live = CatBoostClassifier().load_model(str(ROOT / "models" / run_id / "model.cbm"))
threshold = meta["threshold"]

proba_test = live.predict_proba(X_test[meta["features"]])[:, 1]
pred_test = (proba_test >= threshold).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, pred_test).ravel()

print(f"Live model: {run_id}, threshold {threshold:.4f}")
print(f"Test ROC-AUC {roc_auc_score(y_test, proba_test):.4f} (saved: {meta['metrics']['test']['roc_auc']:.4f})")
cm_table = pd.DataFrame([[tp, fn], [fp, tn]],
                        index=["Has prediabetes/diabetes", "Does not"],
                        columns=["Flagged (see a doctor)", "Not flagged (low risk)"])
cm_table.loc["Total"] = cm_table.sum()
cm_table["Total"] = cm_table.sum(axis=1)
display(cm_table)
print(f"Recall (caught): {tp / (tp + fn):.1%} | precision (flagged who really have it): {tp / (tp + fp):.1%} | "
      f"specificity (healthy not flagged): {tn / (tn + fp):.1%} | flagged in total: {(tp + fp) / len(y_test):.1%}")
print(f"False alarms per true case found: {fp / tp:.2f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
labels = ["No diabetes", "Prediabetes/diabetes"]
ConfusionMatrixDisplay.from_predictions(y_test, pred_test, display_labels=labels, values_format=",",
                                        cmap="Blues", colorbar=False, ax=axes[0])
axes[0].set_title("Counts (test set)")
ConfusionMatrixDisplay.from_predictions(y_test, pred_test, display_labels=labels, normalize="true",
                                        values_format=".1%", cmap="Blues", colorbar=False, ax=axes[1])
axes[1].set_title("Share of each true class")
for ax in axes:
    ax.set_xlabel("Predicted (flagged = 1)"); ax.set_ylabel("True")
plt.tight_layout(); plt.show()

**How to read it:** the rows are the truth, the columns what the model says. The bottom-left cell (false alarms) is
large **by design**: a false alarm costs a doctor's visit, a missed case can go undiagnosed for years (D-23). That
is also why accuracy (~71%) is lower than the dummy's 84%, which flags no one.

## 2. Outliers: Z-score and IQR

18 of the 21 features are 0/1 flags or bounded codes (e.g. `Age` 1–13, `GenHlth` 1–5); they **cannot** have
outliers beyond the allowed codes, and those are already checked by `ml/validation.py` (EDA section 6). Only the
three numeric features are examined here: `BMI`, `MentHlth` and `PhysHlth` (days in the last 30).

Two methods from the course (Session 3, EDA):
- **Z-score:** flag $|z| > 3$ (more than 3 standard deviations from the mean). Works best for roughly symmetric data.
- **IQR:** flag values below $Q1 - k \cdot IQR$ or above $Q3 + k \cdot IQR$, with the usual $k = 1.5$ and the
  stricter $k = 3$ used in the course notebook.

Computed on the **training rows** only (the rule would be fitted there in a pipeline).

In [ ]:
num_cols = ["BMI", "MentHlth", "PhysHlth"]
rows = []
for col in num_cols:
    s = X_train[col]
    z = (s - s.mean()) / s.std()
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    rows.append({
        "Feature": col, "Mean": s.mean(), "Median": s.median(), "Q1": q1, "Q3": q3, "Max": s.max(),
        "Skewness": s.skew(),
        "Z > 3: limit": s.mean() + 3 * s.std(), "Z > 3: share": (z.abs() > 3).mean(),
        "IQR 1.5: upper": q3 + 1.5 * iqr, "IQR 1.5: share": ((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).mean(),
        "IQR 3: upper": q3 + 3 * iqr, "IQR 3: share": ((s < q1 - 3 * iqr) | (s > q3 + 3 * iqr)).mean(),
        "Share of zeros": (s == 0).mean(),
    })
outliers = pd.DataFrame(rows).set_index("Feature")
display(outliers.style.format("{:.3f}").format("{:.1%}", subset=[c for c in outliers if "share" in c.lower()]))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for ax, col in zip(axes, num_cols):
    ax.boxplot(X_train[col], vert=False, whis=1.5, flierprops={"markersize": 2, "alpha": 0.3})
    ax.set_title(col); ax.set_yticks([])
plt.suptitle("Box plots (whiskers = 1.5 × IQR), training rows"); plt.tight_layout(); plt.show()

### Are the BMI outliers errors or real people?

If extreme BMI values were typing errors, they would carry no signal. If they are real, the diabetes rate should
rise with them, as BMI is a known risk factor.

In [ ]:
s = X_train["BMI"]
q1, q3 = s.quantile([0.25, 0.75]); upper = q3 + 1.5 * (q3 - q1)
bands = pd.cut(s, [0, 18.5, 25, 30, upper, 50, 70, 99], right=False,
               labels=["<18.5", "18.5–25", "25–30", f"30–{upper:.0f}", f"{upper:.0f}–50 (IQR outlier)",
                       "50–70 (IQR outlier)", "70–98 (IQR outlier)"])
bmi_bands = (pd.DataFrame({"band": bands, "y": y_train}).groupby("band", observed=True)["y"]
             .agg(People="size", Diabetes_rate="mean"))
bmi_bands["Share of people"] = bmi_bands["People"] / len(s)
display(bmi_bands.style.format({"Diabetes_rate": "{:.1%}", "Share of people": "{:.2%}", "People": "{:,}"}))
print(f"Diabetes rate: IQR outliers {y_train[s > upper].mean():.1%} vs. the rest {y_train[s <= upper].mean():.1%}")

In [ ]:
# MentHlth / PhysHlth: are the "outliers" a real group (many bad days) rather than errors?
for col in ["MentHlth", "PhysHlth"]:
    grp = pd.cut(X_train[col], [-1, 0, 14, 29, 30], labels=["0 days", "1–14", "15–29", "30 (every day)"])
    t = pd.DataFrame({"g": grp, "y": y_train}).groupby("g", observed=True)["y"].agg(["size", "mean"])
    print(col, " | ".join(f"{i}: {r['size'] / len(grp):.1%} of people, {r['mean']:.1%} diabetes"
                          for i, r in t.iterrows()))

**Interpretation:** see the conclusion at the end of the notebook; the numbers are recorded in `RESULTS.md`
(step 13).

## 3. Feature selection: ANOVA F-test

**ANOVA F-test** (`f_classif`, Session 5–6): for each feature, are its mean values significantly different between
people with and without (pre)diabetes? A high F-score and a low p-value mean a strong difference. It is a
**univariate** test: it looks at one feature at a time and only at differences in the mean (a linear signal).

Computed on the training rows only.

In [ ]:
F, p = f_classif(X_train, y_train)
anova = (pd.DataFrame({"F-score": F, "p-value": p}, index=X_train.columns)
         .sort_values("F-score", ascending=False))
anova["Rank"] = range(1, len(anova) + 1)
anova["Spearman with target"] = [X_train[c].corr(y_train, method="spearman") for c in anova.index]
display(anova.style.format({"F-score": "{:,.0f}", "p-value": "{:.1e}", "Spearman with target": "{:+.3f}"}))
print(f"Features with p < 0.001: {(anova['p-value'] < 0.001).sum()} of {len(anova)}")

anova["F-score"].sort_values().plot.barh(figsize=(8, 6), title="ANOVA F-score per feature (training rows)")
plt.xlabel("F-score"); plt.tight_layout(); plt.show()

### 3.1 Would fewer questions be enough?

The F-test says which features differ the most on their own, but not whether the weaker ones still help **in
combination**. So we train CatBoost (default settings, D-21) on the top-*k* features by F-score and compare with
5-fold cross-validation on the training rows. Fewer questions would make the self-check quicker to fill in.

In [ ]:
ranked = list(anova.index)
k_results = {}
for k in [5, 10, 15, 21]:
    start = time.time()
    cols = ranked[:k]
    oof = oof_proba(BASE_PARAMS, cols)
    k_results[k] = summary(y_train, oof) | {"Features": ", ".join(cols) if k <= 10 else f"top {k}",
                                            "CV time (s)": time.time() - start}
    print(f"k={k:2d}: ROC-AUC {k_results[k]['ROC-AUC']:.4f} ({time.time() - start:.0f} s)")
k_table = pd.DataFrame(k_results).T
k_table.index.name = "Top-k features"
display(k_table)

### 3.2 Feature engineering: do combined features help?

Six features built from the 21 answers (D-84): number of conditions (`HighBP + HighChol + Stroke +
HeartDiseaseorAttack`), total bad days (`MentHlth + PhysHlth`), a lifestyle score (`PhysActivity + Fruits + Veggies −
Smoker − HvyAlcoholConsump`) and three interactions (`HighBP × HighChol`, `BMI × Age`, `GenHlth × Age`). Compared
with 5-fold CV on the training rows, for CatBoost (default settings, D-21) and for logistic regression (D-25), which
cannot learn interactions by itself.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import brier_score_loss

def engineer(d):
    d = d.copy()
    d["n_conditions"] = d[["HighBP", "HighChol", "Stroke", "HeartDiseaseorAttack"]].sum(axis=1)
    d["bad_days"] = d["MentHlth"] + d["PhysHlth"]
    d["lifestyle_score"] = d["PhysActivity"] + d["Fruits"] + d["Veggies"] - d["Smoker"] - d["HvyAlcoholConsump"]
    d["HighBP_x_HighChol"] = d["HighBP"] * d["HighChol"]
    d["BMI_x_Age"] = d["BMI"] * d["Age"]
    d["GenHlth_x_Age"] = d["GenHlth"] * d["Age"]
    return d

NEW = ["n_conditions", "bad_days", "lifestyle_score", "HighBP_x_HighChol", "BMI_x_Age", "GenHlth_x_Age"]

def make_lr(extra_numeric):
    pre = ColumnTransformer([("num", StandardScaler(), ["BMI", "MentHlth", "PhysHlth"] + extra_numeric),
                             ("ord", OneHotEncoder(handle_unknown="ignore"), ["GenHlth", "Age", "Education", "Income"])],
                            remainder="passthrough")
    return make_pipeline(pre, LogisticRegression(max_iter=2000))

fe_rows = []
for model_name in ["Logistic regression", "CatBoost"]:
    for variant, data, extra in [("21 features", X_train, []), ("21 + 6 engineered", engineer(X_train), NEW)]:
        start = time.time()
        oof = np.zeros(len(y_train))
        for tr, va in cv.split(data, y_train):
            m = make_lr(extra) if model_name == "Logistic regression" else make_model()
            m.fit(data.iloc[tr], y_train.iloc[tr])
            oof[va] = m.predict_proba(data.iloc[va])[:, 1]
        fe_rows.append({"Model": model_name, "Features": variant, **summary(y_train, oof),
                        "Brier": brier_score_loss(y_train, oof), "CV time (s)": time.time() - start})
fe_table = pd.DataFrame(fe_rows).set_index(["Model", "Features"])
display(fe_table.style.format(precision=4))

## 4. Hyperparameter tuning: GridSearchCV

The model comparison used default settings on purpose (D-21): 500 trees, learning rate 0.1, depth 6. Here we check
whether tuning helps, with `GridSearchCV` as in the course (Session 5–6). We use **stratified** 5-fold CV on the
training rows instead of `TimeSeriesSplit`, because our rows are survey respondents, not a time series.

| Hyperparameter | Values | What it controls |
|---|---|---|
| `depth` | 4, 6, 8 | How many questions each tree may ask (more = more complex interactions) |
| `learning_rate` | 0.05, 0.1 | How much each new tree corrects the previous ones |
| `iterations` | 500, 1000 | Number of trees (a lower learning rate usually needs more) |
| `l2_leaf_reg` | 1, 3, 10 | Regularisation: penalises extreme leaf values, against overfitting |

36 combinations × 5 folds = 180 models. Scored on ROC-AUC (the main comparison metric, D-26). The default
setting is part of the grid, so the search can only find something equal or better on CV.

In [ ]:
param_grid = {"depth": [4, 6, 8], "learning_rate": [0.05, 0.1], "iterations": [500, 1000],
              "l2_leaf_reg": [1, 3, 10]}
grid = GridSearchCV(make_model(), param_grid, scoring="roc_auc", cv=cv, n_jobs=1, refit=False,
                    return_train_score=True, verbose=1)
start = time.time()
grid.fit(X_train, y_train)
print(f"Grid search took {(time.time() - start) / 60:.1f} minutes")

res = pd.DataFrame(grid.cv_results_)
res = res[["param_depth", "param_learning_rate", "param_iterations", "param_l2_leaf_reg",
           "mean_test_score", "std_test_score", "mean_train_score", "mean_fit_time", "rank_test_score"]]
res.columns = ["depth", "learning_rate", "iterations", "l2_leaf_reg", "CV ROC-AUC", "CV std",
               "Train ROC-AUC", "Fit time (s)", "Rank"]
res["Train − CV (overfitting)"] = res["Train ROC-AUC"] - res["CV ROC-AUC"]
res = res.sort_values("Rank")
default_mask = ((res["depth"] == 6) & (res["learning_rate"] == 0.1) & (res["iterations"] == 500)
                & (res["l2_leaf_reg"] == 3))
default_row = res[default_mask].iloc[0]
best_row = res.iloc[0]
print(f"Default (depth 6, lr 0.1, 500 trees, l2 3): CV ROC-AUC {default_row['CV ROC-AUC']:.4f}, "
      f"rank {int(default_row['Rank'])} of {len(res)}")
print(f"Best: {grid.best_params_} -> CV ROC-AUC {grid.best_score_:.4f} "
      f"(+{grid.best_score_ - default_row['CV ROC-AUC']:.4f} over the default)")
print(f"Spread of all {len(res)} combinations: {res['CV ROC-AUC'].min():.4f} – {res['CV ROC-AUC'].max():.4f}")
display(res.head(10).style.format(precision=4))
display(res.tail(5).style.format(precision=4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for (lr, it), g in res.groupby(["learning_rate", "iterations"]):
    g = g.groupby("depth")["CV ROC-AUC"].max()
    axes[0].plot(g.index, g.values, marker="o", label=f"lr {lr}, {it} trees")
axes[0].set_xlabel("depth"); axes[0].set_ylabel("CV ROC-AUC (best l2_leaf_reg)"); axes[0].legend()
axes[0].set_title("Validation score by depth")
axes[1].scatter(res["Train − CV (overfitting)"], res["CV ROC-AUC"], s=18)
axes[1].scatter(default_row["Train − CV (overfitting)"], default_row["CV ROC-AUC"], color="tab:orange",
                s=60, label="default (D-21)")
axes[1].scatter(best_row["Train − CV (overfitting)"], best_row["CV ROC-AUC"], color="tab:red", s=60,
                marker="*", label="best")
axes[1].set_xlabel("Train ROC-AUC − CV ROC-AUC (larger = more overfitting)"); axes[1].set_ylabel("CV ROC-AUC")
axes[1].legend(); axes[1].set_title("Overfitting vs. validation score")
plt.tight_layout(); plt.show()

### 4.1 Best setting vs. default on the test set

The best setting is evaluated the same way as every other model: threshold for 80% recall on its own out-of-fold
predictions, then one run on the test set. The default row uses the live model's numbers from section 1.

In [ ]:
best_params = {k: grid.best_params_[k] for k in ["iterations", "learning_rate", "depth", "l2_leaf_reg"]}
start = time.time()
oof_best = oof_proba(best_params, list(X_train.columns))
thr_best = threshold_for_recall(y_train, oof_best)
tuned = make_model(best_params).fit(X_train, y_train)
proba_best = tuned.predict_proba(X_test)[:, 1]
print(f"Tuned model: CV + final fit took {time.time() - start:.0f} s")

compare = pd.DataFrame({
    "Default (live model)": summary(y_test, proba_test, threshold),
    f"Tuned {best_params}": summary(y_test, proba_best, thr_best),
}).T
compare["False alarms"] = [int(((p >= t) & (y_test == 0)).sum())
                           for p, t in [(proba_test, threshold), (proba_best, thr_best)]]
display(compare.style.format(precision=4))
print(f"Test ROC-AUC difference: {compare['ROC-AUC'].iloc[1] - compare['ROC-AUC'].iloc[0]:+.4f}")

## 5. Conclusions

The interpretation of these results, and what was decided because of them, is in `RESULTS.md` (step 13) and
`DECISIONS.md` (D-82 to D-86).